# 04 · So sánh các mô hình học máy

**Câu hỏi.** AEGIS dùng một *head tuyến tính* trên embedding đóng băng của CLIP và DINOv2. Một mô hình phi tuyến
có cho chuyên gia tốt hơn không, nhất là khi ảnh bị nén JPEG, làm mờ, hoặc đến từ generator chưa từng thấy?

**Hai mô hình so sánh** với `linear_head` (head gốc của AEGIS):

| Mô hình | Nhóm | Ý tưởng |
|---|---|---|
| **RBF-SVM** | kernel | tìm siêu phẳng lề cực đại trong không gian đặc trưng phi tuyến (kernel Gauss), trên PCA-128 |
| **Random Forest** | ensemble cây | 300 cây quyết định trên các mẫu bootstrap và tập đặc trưng ngẫu nhiên, bỏ phiếu đa số |

**Ba phần so sánh**

1. *Tầng embedding*: RBF-SVM và Random Forest thay cho head tuyến tính, trên 3 bộ đặc trưng (CLIP, DINOv2,
   CLIP+DINOv2 ghép nối).
2. *Tầng fusion*: hai mô hình đó dùng làm meta-classifier trên vector 12 chiều, so với Static LR và gating AEGIS.
3. *Không giám sát*: PCA/t-SNE, K-means, phát hiện ảnh lạ (OOD) bằng Isolation Forest / One-Class SVM.

**Giao thức công bằng.** Mọi mô hình huấn luyện trên `train_experts`, chọn siêu tham số theo AUROC trên
`val_gating_fit`, hiệu chỉnh xác suất (Platt) trên chính split đó, rồi đánh giá **một lần** trên `test_benchmark`
và `s4_eval`. Khoảng tin cậy 95% dùng bootstrap ghép cặp theo ảnh gốc, so với `linear_head` (head của AEGIS).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

from src.core.config import load_config
from src.core.contracts import SplitManifest

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 40)
cfg = load_config()
print("repo:", ROOT)

In [ ]:
# Bộ dữ liệu thật (GenImage đã chuẩn hoá). Nếu chưa có, notebook tự chuyển sang DEMO MODE:
# ảnh tổng hợp + embedding giả lập, chạy trên CPU trong ~1 phút. Số liệu DEMO không có ý nghĩa khoa học.
MANIFEST = ROOT / "data/normalized_v2/manifest.json"
DEMO = not MANIFEST.exists()
if DEMO:
    from src.models.demo_features import build_demo
    cfg, man = build_demo(ROOT / "data/demo", scale=6, cfg=cfg)
    MANIFEST = ROOT / "data/demo/images/manifest.json"
else:
    man = SplitManifest.load(MANIFEST)
print("DEMO MODE" if DEMO else "REAL DATA", "|", len(man.records), "records |", MANIFEST)

In [ ]:
from src.evaluation.ml_comparison import (FEATURE_SETS, HEAD_VARIANTS, compare_on_embeddings, load_all,
                                          run_comparison, run_fusion_comparison, write_outputs)
from src.learning.classic_models import DEFAULT_MODELS, REFERENCE, describe_models

FEATURES = list(FEATURE_SETS)          # ["clip", "dinov2", "clip+dinov2"]
HEADS = list(HEAD_VARIANTS)            # ["clean", "aug"]
MODELS = list(DEFAULT_MODELS)          # ["linear_head", "rbf_svm", "random_forest"]
# Muốn so sánh thêm mô hình khác trong bộ: MODELS += ["logreg", "knn", "mlp", ...]  (xem ALL_MODELS)
B = 300 if DEMO else 2000
OUT = ROOT / ("reports/demo_ml_comparison" if DEMO else "reports/ml_comparison")
display(pd.DataFrame(describe_models(MODELS)))

## 1. Huấn luyện, chọn siêu tham số, đánh giá

In [ ]:
out = run_comparison(cfg, man, FEATURES, HEADS, MODELS, out_dir=OUT, B=B)
res, sig = pd.DataFrame(out["results"]), pd.DataFrame(out["significance"])
tuning, timing = pd.DataFrame(out["tuning"]), pd.DataFrame(out["timing"])
print(len(res), "dòng kết quả ->", OUT)

### Bảng chính: AUROC theo kịch bản (S2/S3 gộp mọi mức; S4 chưa thích nghi)

In [ ]:
pooled = res[res.level.isin(["clean", "all"])].copy()
pooled["col"] = np.where(pooled.scenario == "S4", "S4 " + pooled.generator, pooled.scenario)
order = ["S1", "S2", "S3"] + sorted(c for c in pooled.col.unique() if c.startswith("S4"))
main_tab = pooled.pivot_table(index=["head_variant", "feature_set", "model"], columns="col", values="auroc")[order]
display(main_tab.style.format("{:.3f}").background_gradient(axis=None, cmap="RdYlGn"))

### Mô hình tốt nhất cho từng (biến thể head, bộ đặc trưng, kịch bản)

In [ ]:
best = pooled.loc[pooled.groupby(["head_variant", "feature_set", "col"]).auroc.idxmax()]
display(best.pivot_table(index=["head_variant", "feature_set"], columns="col", values="model", aggfunc="first")[order])

## 2. Biểu đồ AUROC theo kịch bản

In [ ]:
HV, FS = HEADS[0], FEATURES[-1]
sub = pooled[(pooled.head_variant == HV) & (pooled.feature_set == FS)]
piv = sub.pivot_table(index="model", columns="col", values="auroc")[order].loc[MODELS]
ax = piv.plot.bar(figsize=(13, 4), width=0.8)
ax.set_ylim(max(0.4, piv.values.min() - 0.05), 1.0); ax.set_ylabel("AUROC")
ax.set_title(f"AUROC theo kịch bản · head={HV} · features={FS}"); ax.legend(ncol=len(order), fontsize=8)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 3. Độ bền dưới từng mức suy giảm (S2 JPEG, S3 blur/resize)

In [ ]:
lv = res[(res.head_variant == HV) & (res.feature_set == FS) & res.scenario.isin(["S1", "S2", "S3"])
         & (res.level != "all")]
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
palette = dict(zip(MODELS, plt.cm.tab20(np.linspace(0, 1, 20))[::max(1, 20 // len(MODELS))]))
for ax, scen in zip(axes, ["S2", "S3"]):
    d = lv[lv.scenario.isin(["S1", scen])]
    levels = ["clean"] + sorted(d[d.scenario == scen].level.unique())
    for m in MODELS:
        y = d[d.model == m].set_index("level").auroc.reindex(levels)
        ax.plot(levels, y.values, marker="o", lw=2.5 if m == REFERENCE else 1.2,
                color="black" if m == REFERENCE else palette[m], label=m)
    ax.set_title(f"{scen} · head={HV} · {FS}"); ax.tick_params(axis="x", rotation=30); ax.grid(alpha=0.3)
axes[0].set_ylabel("AUROC"); axes[1].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 4. Kiểm định ý nghĩa thống kê: ΔAUROC so với `linear_head`

Điểm là ΔAUROC = mô hình − linear_head; thanh ngang là khoảng tin cậy 95% (bootstrap theo ảnh gốc).
Khoảng không cắt 0 nghĩa là khác biệt có ý nghĩa.

In [ ]:
s = sig[(sig.metric == "auroc") & sig.level.isin(["clean", "all"]) & (sig.head_variant == HV) & (sig.feature_set == FS)].copy()
s["col"] = np.where(s.scenario == "S4", "S4 " + s.generator, s.scenario)
cols = [c for c in order if c in set(s.col)]
fig, axes = plt.subplots(1, len(cols), figsize=(3.2 * len(cols), 4.5), sharey=True)
models_ = [m for m in MODELS if m != REFERENCE]
for ax, c in zip(np.atleast_1d(axes), cols):
    d = s[s.col == c].set_index("model").reindex(models_)
    ypos = np.arange(len(d))
    for y_, (_, r) in zip(ypos, d.iterrows()):
        col = "gray" if not r.significant else ("tab:green" if r.delta > 0 else "tab:red")
        ax.errorbar(r.delta, y_, xerr=[[r.delta - r.ci_lo], [r.ci_hi - r.delta]], fmt="o", color=col, capsize=3)
    ax.axvline(0, color="black", lw=0.8); ax.set_title(c); ax.set_yticks(ypos, models_)
fig.suptitle(f"ΔAUROC so với {REFERENCE} · head={HV} · {FS} (xanh: tốt hơn có ý nghĩa, đỏ: kém hơn)")
plt.tight_layout(); plt.show()
display(s[["col", "model", "model_value", "reference_value", "delta", "ci_lo", "ci_hi", "p_value", "significant"]]
        .sort_values(["col", "delta"], ascending=[True, False]))

## 5. Ghép CLIP+DINOv2 (early fusion) có tốt hơn từng backbone riêng lẻ?

In [ ]:
fs_tab = pooled[pooled.head_variant == HV].pivot_table(index="model", columns=["col", "feature_set"], values="auroc")
display(fs_tab[[c for c in order if c in fs_tab.columns.get_level_values(0)]].loc[MODELS].style.format("{:.3f}"))

## 6. Chi phí tính toán: thời gian huấn luyện, thời gian dự đoán và AUROC

In [ ]:
t = timing[(timing.head_variant == HV) & (timing.feature_set == FS)].merge(
    pooled[(pooled.head_variant == HV) & (pooled.feature_set == FS) & (pooled.col == "S1")][["model", "auroc"]], on="model")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(t.fit_seconds, t.auroc)
for _, r in t.iterrows():
    ax.annotate(r.model, (r.fit_seconds, r.auroc), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("thời gian fit (s, log)"); ax.set_ylabel("AUROC S1")
ax.set_title(f"Chi phí và hiệu năng · head={HV} · {FS}"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
display(t[["model", "params", "val_auroc", "fit_seconds", "predict_ms_per_1k", "auroc"]].sort_values("auroc", ascending=False))

## 7. Tầng fusion: các mô hình làm meta-classifier trên vector gating 12 chiều

Đầu vào: chất lượng ảnh (3) | entropy (3) | khoảng cách OOD (3) | xác suất đã hiệu chỉnh của 3 chuyên gia (3).
Fit trên `val_gating_fit`, chọn siêu tham số trên `val_gating_verify`, đánh giá trên test. Đối chứng là
`static_lr` (trọng số cố định) và `aegis` (gating theo từng ảnh). S4 ở đây là *chưa thích nghi few-shot*.

In [ ]:
fus = pd.DataFrame(sum((run_fusion_comparison(cfg, hv, MODELS) for hv in HEADS), []))
fus_p = fus[fus.level.isin(["clean", "all"])]
display(fus_p.pivot_table(index=["head_variant", "method"], columns="scenario", values="auroc")
        .style.format("{:.3f}").background_gradient(axis=None, cmap="RdYlGn"))
fus.to_csv(OUT / "fusion_level.csv", index=False)

## 8. Học không giám sát trên embedding

* **PCA / t-SNE**: embedding có tự tách Real và AI, hay tách theo generator không?
* **K-means**: cụm có trùng với generator không (Adjusted Rand Index, purity)?
* **Phát hiện ảnh lạ**: chỉ học trên ảnh sạch của generator đã thấy, có nhận ra ảnh S4 (Midjourney, VQDM) là lạ không?
  So sánh Isolation Forest, One-Class SVM với khoảng cách cosine tới tâm (cách AEGIS đang dùng).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
data = load_all(cfg, man, FS, "clean")
te = data["test"]
m_clean = np.array([s is None for s in te.scenario])
Xv = np.concatenate([te.X[m_clean], data["s4"].X])
gv = np.array([g for g, c in zip(te.generator, m_clean) if c] + list(data["s4"].generator))
Z = StandardScaler().fit_transform(Xv)
P2 = PCA(2, random_state=0).fit_transform(Z)
idx = np.random.default_rng(0).choice(len(Z), min(2000, len(Z)), replace=False)
T2 = TSNE(2, random_state=0, init="pca", perplexity=min(30, len(idx) // 4)).fit_transform(Z[idx])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, E, gg, title in ((axes[0], P2, gv, "PCA"), (axes[1], T2, gv[idx], "t-SNE")):
    for g in sorted(set(gg)):
        k = gg == g
        ax.scatter(E[k, 0], E[k, 1], s=6, alpha=0.6, label=g, c="black" if g == "real" else None)
    ax.set_title(f"{title} · {FS} · test sạch + s4_eval"); ax.legend(markerscale=3, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
Zp = PCA(min(50, Z.shape[1]), random_state=0).fit_transform(Z)
rows = []
for k in (2, len(set(gv))):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Zp)
    target = gv if k > 2 else (gv != "real").astype(int)
    purity = pd.crosstab(lab, target).max(axis=1).sum() / len(lab)
    rows.append({"k": k, "so với": "generator" if k > 2 else "Real/AI", "ARI": adjusted_rand_score(target, lab),
                 "NMI": normalized_mutual_info_score(target, lab), "purity": purity})
display(pd.DataFrame(rows))

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.svm import OneClassSVM
tr = data["train"]
sc = StandardScaler().fit(tr.X)
pca = PCA(min(64, tr.X.shape[1]), random_state=0).fit(sc.transform(tr.X))
f = lambda X: pca.transform(sc.transform(X))
s4_ai = data["s4"].X[data["s4"].y == 1]
seen = te.X[m_clean]
X_ood = np.concatenate([seen, s4_ai]); y_ood = np.r_[np.zeros(len(seen)), np.ones(len(s4_ai))]
centroid = tr.X.mean(0)
cosd = 1 - (X_ood @ centroid) / (np.linalg.norm(X_ood, axis=1) * np.linalg.norm(centroid) + 1e-9)
scores = {
    "cosine distance (AEGIS)": cosd,
    "Isolation Forest": -IsolationForest(n_estimators=300, random_state=0).fit(f(tr.X)).score_samples(f(X_ood)),
    "One-Class SVM": -OneClassSVM(nu=0.1, gamma="scale").fit(f(tr.X)).score_samples(f(X_ood)),
}
display(pd.DataFrame({"AUROC phát hiện ảnh S4 là lạ": {k: roc_auc_score(y_ood, v) for k, v in scores.items()}}))

## 9. Gợi ý viết phần thảo luận trong báo cáo

* Mô hình nào thắng ở ảnh sạch (S1)? Thắng có ý nghĩa thống kê không, hay mọi mô hình đều gần trần?
* Dưới JPEG/blur (S2/S3), RBF-SVM và Random Forest bền hơn `linear_head` hay overfit vào ảnh sạch?
  So sánh head `clean` và `aug`: tăng cường dữ liệu và đổi mô hình, cách nào có lợi hơn?
* S4 (generator chưa thấy): Random Forest (dựa trên các ngưỡng cắt học từ generator đã thấy) có tổng quát hoá kém
  hơn mô hình tuyến tính và RBF-SVM không?
* Ghép CLIP+DINOv2 (early fusion) so với gating AEGIS (late fusion): bên nào tốt hơn, ở kịch bản nào?
* Chi phí: RBF-SVM và Random Forest chậm hơn `linear_head` bao nhiêu lần, và phần AUROC tăng thêm có đáng không?
* Luôn so với mức sàn `quality_only` (≈0.63 trên dữ liệu thật) để chắc mô hình không học shortcut.